# Create Knowledge Assistant for Logistics Control Center

This notebook resolves or creates a Knowledge Assistant (KA) endpoint for document-based Q&A.

**What is Knowledge Assistant?**
Knowledge Assistant is a Databricks feature that indexes documents and enables natural language Q&A over them.

**Strategy:**
1. First, try to find an existing KA endpoint (preferred explicit endpoint first)
2. If Agent Bricks API is available, attempt to create a new KA
3. If creation APIs are unavailable, provide remediation instructions

**Documents Indexed:**
- Incident Analysis Reports
- Maintenance Bulletins
- Standard Operating Procedures
- Customer SLA Documents
- Route Planning Guides
- Root Cause Analysis Reports

## Configuration

Load configuration from job parameters, environment variables, or use defaults.

In [ ]:
from __future__ import annotations

import os
from typing import Any, Iterable, Optional


def _get_config(name: str, default: str) -> str:
    """Get config from job parameters (widgets), env vars, or default."""
    try:
        return dbutils.widgets.get(name).strip()  # type: ignore[name-defined]
    except Exception:
        return os.environ.get(f"DATABRICKS_{name.upper()}", default).strip()


PREFERRED_ENDPOINT = _get_config("ka_endpoint", "ka-3c254141-endpoint")
KA_NAME = _get_config("ka_name", "logistics-control-center-knowledge-assistant")
CATALOG = _get_config("catalog", "main")
SCHEMA = _get_config("schema", "logistics_control_center")
DOCS_VOLUME = _get_config("ka_docs_volume", "documents")
DOCS_VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{DOCS_VOLUME}"

print(f"Configuration:")
print(f"  Preferred Endpoint: {PREFERRED_ENDPOINT}")
print(f"  KA Name: {KA_NAME}")
print(f"  Documents Path: {DOCS_VOLUME_PATH}")

## Initialize Databricks SDK Client

Connect to the workspace using the notebook's authentication context.

In [ ]:
from databricks.sdk import WorkspaceClient

client = WorkspaceClient()
print(f"Connected to workspace: {client.config.host}")

## Helper Functions

Utility functions for finding and creating Knowledge Assistant endpoints.

In [ ]:
def _non_empty_names(names: Iterable[Optional[str]]) -> list[str]:
    """Filter out None and empty strings from endpoint names."""
    return [name for name in names if isinstance(name, str) and name.strip()]


def _existing_ka_endpoint(endpoints: list[str], preferred: str) -> Optional[str]:
    """Find an existing KA endpoint, preferring the specified one."""
    if preferred and preferred in endpoints:
        return preferred
    ka_like = [name for name in endpoints if name.startswith("ka-") or "knowledge" in name.lower()]
    return sorted(ka_like)[0] if ka_like else None


def _ready_state(client: WorkspaceClient, endpoint_name: str) -> Optional[str]:
    """Check if an endpoint is ready."""
    try:
        endpoint = client.serving_endpoints.get(endpoint_name)
    except Exception:
        return None
    state = getattr(endpoint, "state", None)
    return str(getattr(state, "ready", "")) if state else None

## KA Creation Function (Agent Bricks API)

Attempt to create a Knowledge Assistant using the Agent Bricks API if available. This API may not be present in all runtime versions.

In [ ]:
def _maybe_create_ka(client: WorkspaceClient, ka_name: str, volume_path: str) -> Optional[str]:
    """Best-effort KA create path for runtimes exposing Agent Bricks KA APIs."""
    api = getattr(client, "agent_bricks", None)
    if api is None:
        print("Agent Bricks API not available in this runtime.")
        return None

    # Check for required methods
    has_find = hasattr(api, "find_ka_by_name")
    has_create_or_update = hasattr(api, "create_or_update_ka")
    if not (has_find and has_create_or_update):
        print("Agent Bricks API missing required KA methods.")
        return None

    # Try to find existing KA by name
    found = api.find_ka_by_name(name=ka_name)
    if isinstance(found, dict) and found.get("found") and found.get("endpoint_name"):
        return str(found["endpoint_name"])
    if getattr(found, "found", False) and getattr(found, "endpoint_name", None):
        return str(found.endpoint_name)

    # Create new KA
    print(f"Creating new Knowledge Assistant: {ka_name}")
    created = api.create_or_update_ka(
        name=ka_name,
        volume_path=volume_path,
        description="Logistics operations knowledge assistant for incident, SOP, and SLA docs.",
        instructions=(
            "You are a logistics operations assistant. Answer from indexed documents, "
            "cite relevant sources, and highlight actionable incident-response guidance."
        ),
    )
    
    # Extract endpoint name from response
    endpoint_name = getattr(created, "endpoint_name", None)
    if endpoint_name:
        return str(endpoint_name)

    if isinstance(created, dict):
        endpoint_name = created.get("endpoint_name")
        if endpoint_name:
            return str(endpoint_name)
        tile_id = created.get("tile_id") or created.get("id")
        if tile_id:
            return f"ka-{tile_id}-endpoint"

    tile_id = getattr(created, "tile_id", None) or getattr(created, "id", None)
    if tile_id:
        return f"ka-{tile_id}-endpoint"

    return None

## List Available Serving Endpoints

Get all serving endpoints to check for existing KA endpoints.

In [ ]:
endpoint_names = _non_empty_names(getattr(ep, "name", None) for ep in client.serving_endpoints.list())

ka_endpoints = [name for name in endpoint_names if name.startswith("ka-") or "knowledge" in name.lower()]
print(f"Found {len(ka_endpoints)} KA-like endpoints:")
for ep in ka_endpoints[:10]:  # Show first 10
    print(f"  - {ep}")

## Resolve or Create Knowledge Assistant

Find an existing KA endpoint or create a new one if the API is available.

In [ ]:
selected = _existing_ka_endpoint(endpoint_names, PREFERRED_ENDPOINT)

if selected:
    print(f"Found existing KA endpoint: {selected}")
else:
    print("No existing KA endpoint found. Attempting to create...")
    selected = _maybe_create_ka(client, KA_NAME, DOCS_VOLUME_PATH)

if not selected:
    error_msg = (
        "Knowledge Assistant endpoint is not available and this runtime does not expose a KA create API.\n"
        "Create the KA manually, then deploy with a bundle variable override:\n"
        '  databricks bundle deploy --target dev --var "ka_endpoint=<ka-endpoint-name>"\n'
        "Also ensure app runtime env sets DATABRICKS_KA_ENDPOINT."
    )
    raise RuntimeError(error_msg)

## Output Results

Display the KA endpoint name and ready state for use in downstream configuration.

In [ ]:
ready = _ready_state(client, selected)

print(f"✓ Resolved Knowledge Assistant endpoint: {selected}")
print(f"  Ready state: {ready or 'unknown'}")
print()
print(f"DATABRICKS_KA_ENDPOINT={selected}")
print()
print("Add this value to your databricks.yml targets.dev.variables section.")